In [10]:
import torch
import pandas as pd
from IPython.display import display
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)

In [11]:
graff_formula_stats_df = pd.read_csv(f"../figs/formula_ann_exp/graff/formula_stats_df.csv")
nograff_formula_stats_df = pd.read_csv(f"../figs/formula_ann_exp/nograff/formula_stats_df.csv")
formula_stats_df = pd.concat([graff_formula_stats_df, nograff_formula_stats_df])
display(formula_stats_df)

In [12]:
# calculate F1 score
def f1_score(recall, precision):
    return 2 / (1 / recall + 1 / precision)

metric_prefixes = ["both_t_peak", "both_t_w_peak", "both_p_peak", "both_p_w_peak"]

formula_stats_df["both_peak_f1_0.0_mean"] = f1_score(formula_stats_df["both_t_peak_cov_0.0_mean"], formula_stats_df["both_p_peak_cov_0.0_mean"])
formula_stats_df["both_w_peak_f1_0.0_mean"] = f1_score(formula_stats_df["both_t_w_peak_cov_0.0_mean"], formula_stats_df["both_p_w_peak_cov_0.0_mean"])

In [13]:
# average by seed
metric_cols = [
    "both_t_peak_cov_0.0_mean",
    "both_t_w_peak_cov_0.0_mean",
    "both_t_peak_cov_2_0.0_mean",
    "both_t_w_peak_cov_2_0.0_mean",
    "mode_peak_cov_mean",
    "mode_w_peak_cov_mean",
    "both_p_peak_cov_0.0_mean",
    "both_p_w_peak_cov_0.0_mean",
    "both_p_peak_cov_2_0.0_mean",
    "both_p_w_peak_cov_2_0.0_mean",
    "both_peak_f1_0.0_mean",
    "both_w_peak_f1_0.0_mean"
]

formula_stats_df = formula_stats_df[["model_type"]+metric_cols].groupby(["model_type"]).agg(["mean","std"]).reset_index()
display(formula_stats_df)

In [14]:
model_to_label = {
    "fraggnn_d4_long": "FraGNNet-D4",
    "graff3_long": "GrAFF",
    "iceberg_inten3_100_long": "ICEBERG"
}

column_to_label = {
    "model_type": "Model",
    "both_t_peak_cov_0.0_mean": "True Ann Peak Coverage",
    "both_t_w_peak_cov_0.0_mean": "True Ann Peak Weighted Coverage",
    "both_t_peak_cov_2_0.0_mean": "True Peak Coverage",
    "both_t_w_peak_cov_2_0.0_mean": "True Peak Weighted Coverage",
    "mode_peak_cov_mean": "True Ann Peak Mode Coverage",
    "mode_w_peak_cov_mean": "True Ann Peak Mode Weighted Coverage",
    "both_p_peak_cov_0.0_mean": "Pred Ann Peak Coverage",
    "both_p_w_peak_cov_0.0_mean": "Pred Ann Peak Weighted Coverage",
    "both_p_peak_cov_2_0.0_mean": "Pred Peak Coverage",
    "both_p_w_peak_cov_2_0.0_mean": "Pred Peak Weighted Coverage",
    "both_peak_f1_0.0_mean": "Ann Peak F1 Score",
    "both_w_peak_f1_0.0_mean": "Ann Peak Weighted F1 Score"
}

formula_stats_df_1 = formula_stats_df.copy(deep=True)
formula_stats_df_1["model_type"] = formula_stats_df_1["model_type"].map(model_to_label)
formula_stats_df_1 = formula_stats_df_1[list(column_to_label.keys())]
formula_stats_df_1 = formula_stats_df_1.rename(columns=column_to_label)
display(formula_stats_df_1)

In [15]:
def flatten_mean_std(df):
    # Select columns that are either not multi-index or are multi-index with "mean" as the second level
    df_means = df.loc[:, [col for col in df.columns if col[1] == 'mean' or col[1] == '']]
    # Rename multi-index columns to just use the first level (the metric name)
    df_means.columns = [col[0] if isinstance(col, tuple) else col for col in df_means.columns]
    return df_means

In [16]:
true_formula_stats_df_1 = flatten_mean_std(formula_stats_df_1[[('Model', '')] + [col for col in formula_stats_df_1.columns if "True" in col[0]]])
pred_formula_stats_df_1 = flatten_mean_std(formula_stats_df_1[[('Model', '')] + [col for col in formula_stats_df_1.columns if "Pred" in col[0]]])
display(true_formula_stats_df_1)
display(pred_formula_stats_df_1)

,Model,True Ann Peak Coverage,True Ann Peak Weighted Coverage,True Peak Coverage,True Peak Weighted Coverage,True Ann Peak Mode Coverage,True Ann Peak Mode Weighted Coverage
0,FraGNNet-D4,0.808787,0.899675,0.704921,0.885377,0.807874,0.899330
1,GrAFF,0.950905,0.980441,0.827303,0.964845,0.784329,0.766186
2,ICEBERG,0.655931,0.809356,0.564112,0.796221,NaN,NaN


,Model,Pred Ann Peak Coverage,Pred Ann Peak Weighted Coverage,Pred Peak Coverage,Pred Peak Weighted Coverage
0,FraGNNet-D4,0.983682,0.999095,0.092035,0.868824
1,GrAFF,0.610631,0.802179,0.009556,0.672753
2,ICEBERG,0.992557,NaN,0.145484,NaN


In [17]:
model_to_label = {
    "fraggnn_d4_long": "FraGNNet-D4",
    "graff3_long": "GrAFF",
    "iceberg_inten3_100_long": "ICEBERG"
}

column_to_label = {
    "model_type": "Model",
    "both_t_peak_cov_0.0_mean": "AR", #"True Ann Peak Coverage",
    "both_t_w_peak_cov_0.0_mean": "AWR", #"True Ann Peak Weighted Coverage",
    # "both_t_peak_cov_2_0.0_mean": "True Peak Coverage",
    # "both_t_w_peak_cov_2_0.0_mean": "True Peak Weighted Coverage",
    # "mode_peak_cov_mean": "True Ann Peak Mode Coverage",
    # "mode_w_peak_cov_mean": "True Ann Peak Mode Weighted Coverage",
    "both_p_peak_cov_0.0_mean": "AP", #"Pred Ann Peak Coverage",
    "both_p_w_peak_cov_0.0_mean": "AWP", #"Pred Ann Peak Weighted Coverage",
    # "both_p_peak_cov_2_0.0_mean": "Pred Peak Coverage",
    # "both_p_w_peak_cov_2_0.0_mean": "Pred Peak Weighted Coverage",
    "both_peak_f1_0.0_mean": "AF1", #"Ann Peak F1 Score",
    "both_w_peak_f1_0.0_mean": "AWF1" #"Ann Peak Weighted F1 Score"
}

formula_stats_df_2 = formula_stats_df.copy(deep=True)
formula_stats_df_2["model_type"] = formula_stats_df_2["model_type"].map(model_to_label)
formula_stats_df_2 = formula_stats_df_2[list(column_to_label.keys())]
formula_stats_df_2 = formula_stats_df_2.rename(columns=column_to_label)
display(formula_stats_df_2)

Model        AR            AWR                      AP       \
                    mean  std      mean           std      mean  std   
0  FraGNNet-D4  0.808787  0.0  0.899675  2.522166e-10  0.983682  0.0   
1        GrAFF  0.950905  0.0  0.980441  8.151887e-10  0.610631  0.0   
2      ICEBERG  0.655931  0.0  0.809356  2.841673e-10  0.992557  0.0   

        AWP                 AF1           AWF1            
       mean       std      mean  std      mean       std  
0  0.999095  0.000236  0.887702  0.0  0.946782  0.000106  
1  0.802179  0.006028  0.743694  0.0  0.882391  0.003647  
2       NaN       NaN  0.789874  0.0       NaN       NaN

In [18]:
metrics = list(column_to_label.values())[1:]

latex_stats_df_2 = formula_stats_df_2.copy(deep=True)

for metric in metrics:
    latex_stats_df_2[metric] = latex_stats_df_2[metric].apply(
        lambda row: "-" if pd.isna(row['mean']) or pd.isna(row['std'])
                    else f"${row['mean']:.2f} \pm {row['std']:.2f}$", 
        axis=1
    )

latex_stats_df_2 = latex_stats_df_2[[col for col in latex_stats_df_2.columns if (col[1] == "" or col[1] == "mean")]]
latex_stats_df_2.columns = [col + r" $\Uparrow$" if col != "Model" else col for col in latex_stats_df_2.columns.get_level_values(0)]

display(latex_stats_df_2)

latex_stats_df_2_str = latex_stats_df_2.to_latex(index=False, escape=False, column_format="lcccc")

with open(f"../figs/formula_ann_exp/formula_stats_df.tex", "w") as f:
    f.write(latex_stats_df_2_str)

,Model,AR $\Uparrow$,AWR $\Uparrow$,AP $\Uparrow$,AWP $\Uparrow$,AF1 $\Uparrow$,AWF1 $\Uparrow$
0,FraGNNet-D4,$0.81 \pm 0.00$,$0.90 \pm 0.00$,$0.98 \pm 0.00$,$1.00 \pm 0.00$,$0.89 \pm 0.00$,$0.95 \pm 0.00$
1,GrAFF,$0.95 \pm 0.00$,$0.98 \pm 0.00$,$0.61 \pm 0.00$,$0.80 \pm 0.01$,$0.74 \pm 0.00$,$0.88 \pm 0.00$
2,ICEBERG,$0.66 \pm 0.00$,$0.81 \pm 0.00$,$0.99 \pm 0.00$,-,$0.79 \pm 0.00$,-
